# 07 - Đánh giá các mô hình Face Anti-Spoofing
Nhãn: real = 0, spoof = 1. Ngưỡng lấy từ `best_threshold.txt` (đã tìm trên val lúc huấn luyện). Chỉ số chính: APCER, BPCER, ACER; thêm EER, AUC, ACC.

In [ ]:
import os
import sys
import math
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

sys.path.append(os.path.abspath('..'))

from models import MultiBranchModel
from src import transforms
from src.evaluate import calculate_metrics

from keras.models import load_model
from keras.utils import image_dataset_from_directory

## 1. Cấu hình

In [ ]:
ROOT = "../outputs"
TEST = "../datasets"
TEST_PATH = ["/LCC_FASD/test", "/CASIA-FASD", "/Oulu-NPU", "/SiW", "/MSU-FASD", "/CelebA_Spoof"]
CLASS = ["real", "spoof"]
PATH_MODEL = ["custom-model", "backbone-model", "multibranch-model", "celebA-model"]
NAME_MODEL = "best_model.keras"
THRESHOLD = "best_threshold.txt"

IMG_SIZE = (224, 224)
BATCH = 32
MAX_SAMPLES = 5000

results = {}
scores = {}

In [ ]:
def predict_model(name, img_size):
    model = model[name]["model"]
    threshold = model[name]["threshold"]

    scores[name] = {}
    rows = []

    for path in TEST_PATH:
        ds = image_dataset_from_directory(
            TEST + path, 
            label_mode="binary", 
            class_names=CLASS,
            image_size=img_size, 
            batch_size=BATCH, 
            shuffle=True, 
            seed=42,
        )
        ds = ds.take(math.ceil(MAX_SAMPLES / BATCH)).prefetch(tf.data.AUTOTUNE)

        ys, ps = [], []
        for x, y in ds:
            ps.append(model.predict_on_batch(x).flatten())
            ys.append(y.numpy().flatten())
        y_true = np.concatenate(ys).astype(int)
        y_prob = np.concatenate(ps)

        scores[name][path] = (y_true, y_prob)
        rows.append({
            "model": name,
            "dataset": path.strip("/"),
            "n_real": int((y_true == 0).sum()),
            "n_spoof": int((y_true == 1).sum()),
            **calculate_metrics(y_true, y_prob, threshold),
        })
        print("xong", path)

    results[name] = pd.DataFrame(rows).round(2)
    results[name].to_csv(f"../outputs/results_{name}.csv", index=False)
    results[name]

## 2. Nạp mô hình và ngưỡng

In [9]:
model = {}

for name in PATH_MODEL:
    model[name] = {
        "model": load_model(Path(ROOT) / name / NAME_MODEL),
        "threshold": float(open(Path(ROOT) / name / THRESHOLD).readline().strip()),
    }

print("Total models loaded:", len(model))
for name in model:
    print(name, "threshold =", model[name]["threshold"])

Total models loaded: 4
custom-model threshold = 0.45997369289398193
backbone-model threshold = 0.7183240652084351
multibranch-model threshold = 0.48678967356681824
celebA-model threshold = 0.6268168


## 3. Kiểm tra dữ liệu trước khi dự đoán

In [10]:
# Đếm số mẫu từng tập
for p in TEST_PATH:
    n = {c: len(list((Path(TEST + p) / c).glob("*"))) for c in CLASS}
    print(p, n)

/LCC_FASD/test {'real': 314, 'spoof': 7266}
/CASIA-FASD {'real': 10128, 'spoof': 55658}
/Oulu-NPU {'real': 1358, 'spoof': 343}
/SiW {'real': 600, 'spoof': 150}
/MSU-FASD {'real': 40, 'spoof': 119}
/CelebA_Spoof {'real': 19923, 'spoof': 47247}


In [11]:
# Kiểm tra nhãn và thang pixel
ds = image_dataset_from_directory(TEST + TEST_PATH[0], label_mode="binary", class_names=CLASS,
                                  image_size=IMG_SIZE, batch_size=BATCH)
print(ds.class_names)          # phải là ['real', 'spoof']  ->  real = 0, spoof = 1

Found 7580 files belonging to 2 classes.
['real', 'spoof']


## 4. CustomModel

In [ ]:
predict_model("custom-model", (224, 224))

Found 7580 files belonging to 2 classes.
xong /LCC_FASD/test
Found 65786 files belonging to 2 classes.
xong /CASIA-FASD
Found 1701 files belonging to 2 classes.
xong /Oulu-NPU
Found 750 files belonging to 2 classes.
xong /SiW
Found 44002 files belonging to 2 classes.
xong /MSU-FASD
Found 67170 files belonging to 2 classes.
xong /CelebA_Spoof


,model,dataset,n_real,n_spoof,EER (%),APCER (%),BPCER (%),ACER (%),AUC (%),ACC (%)
0,custom-model,LCC_FASD/test,215,4809,20.93,8.98,35.35,22.17,87.12,89.89
1,custom-model,CASIA-FASD,801,4223,52.18,34.05,76.65,55.35,50.77,59.16
2,custom-model,Oulu-NPU,1358,343,48.60,76.97,24.67,50.82,51.94,64.79
3,custom-model,SiW,600,150,33.33,75.33,9.17,42.25,73.16,77.60
4,custom-model,MSU-FASD,1250,3774,29.84,18.52,52.32,35.42,79.53,73.07
5,custom-model,CelebA_Spoof,1467,3557,33.88,22.38,47.51,34.95,71.12,70.28


## 5. BackboneModel (EfficientNet-B0)

In [ ]:
predict_model("backbone-model", (224, 224))

Found 7580 files belonging to 2 classes.
xong /LCC_FASD/test
Found 65786 files belonging to 2 classes.
xong /CASIA-FASD
Found 1701 files belonging to 2 classes.
xong /Oulu-NPU
Found 750 files belonging to 2 classes.
xong /SiW
Found 44002 files belonging to 2 classes.
xong /MSU-FASD
Found 67170 files belonging to 2 classes.
xong /CelebA_Spoof


,model,dataset,n_real,n_spoof,EER (%),APCER (%),BPCER (%),ACER (%),AUC (%),ACC (%)
0,backbone-model,LCC_FASD/test,215,4809,11.16,12.71,7.91,10.31,95.92,87.50
1,backbone-model,CASIA-FASD,801,4223,41.82,78.71,4.37,41.54,59.87,33.14
2,backbone-model,Oulu-NPU,1358,343,67.45,82.80,51.40,67.10,24.47,42.27
3,backbone-model,SiW,600,150,32.67,92.00,3.00,47.50,73.49,79.20
4,backbone-model,MSU-FASD,1250,3774,36.72,51.70,23.12,37.41,66.21,55.41
5,backbone-model,CelebA_Spoof,1467,3557,32.24,71.77,3.82,37.80,75.15,48.07


## 6. MultiBranchModel

In [ ]:
predict_model("multibranch-model", (224, 224))

Found 7580 files belonging to 2 classes.
xong /LCC_FASD/test
Found 65786 files belonging to 2 classes.
xong /CASIA-FASD
Found 1701 files belonging to 2 classes.
xong /Oulu-NPU
Found 750 files belonging to 2 classes.
xong /SiW
Found 44002 files belonging to 2 classes.
xong /MSU-FASD
Found 67170 files belonging to 2 classes.
xong /CelebA_Spoof


,model,dataset,n_real,n_spoof,EER (%),APCER (%),BPCER (%),ACER (%),AUC (%),ACC (%)
0,multibranch-model,LCC_FASD/test,215,4809,13.02,17.03,10.23,13.63,92.03,83.26
1,multibranch-model,CASIA-FASD,801,4223,46.57,93.11,1.00,47.05,61.51,21.58
2,multibranch-model,Oulu-NPU,1358,343,57.36,73.76,43.45,58.60,38.21,50.44
3,multibranch-model,SiW,600,150,21.33,50.67,11.17,30.92,86.11,80.93
4,multibranch-model,MSU-FASD,1250,3774,23.76,11.45,54.16,32.80,81.45,77.93
5,multibranch-model,CelebA_Spoof,1467,3557,45.19,91.90,4.57,48.24,56.79,33.60


## 7. CelebAModel

In [ ]:
predict_model("celebA-model", (112, 112))

Found 7580 files belonging to 2 classes.


I0000 00:00:1791879510.726811   20219 service.cc:153] XLA service 0x7f2f3c2299d0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791879510.726859   20219 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1791879510.992814   20219 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791879514.267393   20219 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


xong /LCC_FASD/test
Found 65786 files belonging to 2 classes.
xong /CASIA-FASD
Found 1701 files belonging to 2 classes.
xong /Oulu-NPU
Found 750 files belonging to 2 classes.
xong /SiW
Found 44002 files belonging to 2 classes.
xong /MSU-FASD
Found 67170 files belonging to 2 classes.
xong /CelebA_Spoof


,model,dataset,n_real,n_spoof,EER (%),APCER (%),BPCER (%),ACER (%),AUC (%),ACC (%)
0,celebA-model,LCC_FASD/test,215,4809,23.26,30.11,18.14,24.12,85.31,70.40
1,celebA-model,CASIA-FASD,801,4223,35.83,31.02,46.57,38.79,68.12,66.50
2,celebA-model,Oulu-NPU,1358,343,58.32,43.15,66.72,54.93,37.74,38.04
3,celebA-model,SiW,600,150,12.00,22.67,5.00,13.83,95.26,91.47
4,celebA-model,MSU-FASD,1250,3774,21.52,75.78,3.92,39.85,84.96,42.10
5,celebA-model,CelebA_Spoof,1467,3557,9.07,21.59,2.45,12.02,96.93,84.00


## 8. Tổng hợp

In [18]:
df = pd.concat(results.values(), ignore_index=True)
df.to_csv("../outputs/evaluate_summary.csv", index=False)
df

,model,dataset,n_real,n_spoof,EER (%),APCER (%),BPCER (%),ACER (%),AUC (%),ACC (%)
0,custom-model,LCC_FASD/test,215,4809,20.93,8.98,35.35,22.17,87.12,89.89
1,custom-model,CASIA-FASD,801,4223,52.18,34.05,76.65,55.35,50.77,59.16
2,custom-model,Oulu-NPU,1358,343,48.60,76.97,24.67,50.82,51.94,64.79
3,custom-model,SiW,600,150,33.33,75.33,9.17,42.25,73.16,77.60
4,custom-model,MSU-FASD,1250,3774,29.84,18.52,52.32,35.42,79.53,73.07
5,custom-model,CelebA_Spoof,1467,3557,33.88,22.38,47.51,34.95,71.12,70.28
6,backbone-model,LCC_FASD/test,215,4809,11.16,12.71,7.91,10.31,95.92,87.50
7,backbone-model,CASIA-FASD,801,4223,41.82,78.71,4.37,41.54,59.87,33.14
8,backbone-model,Oulu-NPU,1358,343,67.45,82.80,51.40,67.10,24.47,42.27
9,backbone-model,SiW,600,150,32.67,92.00,3.00,47.50,73.49,79.20


In [ ]:
df.pivot(index="dataset", columns="model", values="ACC (%)")

model,backbone-model,celebA-model,custom-model,multibranch-model
dataset,,,,
CASIA-FASD,33.14,66.50,59.16,21.58
CelebA_Spoof,48.07,84.00,70.28,33.60
LCC_FASD/test,87.50,70.40,89.89,83.26
MSU-FASD,55.41,42.10,73.07,77.93
Oulu-NPU,42.27,38.04,64.79,50.44
SiW,79.20,91.47,77.60,80.93


In [21]:
df.pivot(index="dataset", columns="model", values="AUC (%)")

model,backbone-model,celebA-model,custom-model,multibranch-model
dataset,,,,
CASIA-FASD,59.87,68.12,50.77,61.51
CelebA_Spoof,75.15,96.93,71.12,56.79
LCC_FASD/test,95.92,85.31,87.12,92.03
MSU-FASD,66.21,84.96,79.53,81.45
Oulu-NPU,24.47,37.74,51.94,38.21
SiW,73.49,95.26,73.16,86.11
